In [ ]:
import scanpy as sc
import anndata as ad
import os
import pandas as pd
import numpy as np
import seaborn as sns
import skimage 
from itertools import combinations
import matplotlib.pyplot as plt
from pathlib import Path
import yaml



In [ ]:
import scanpy as sc


In [ ]:
 scanpy.logging.print_versions()

In [ ]:
#  Load config 
cfg = yaml.safe_load(open("configs/config.yaml"))
DATA_ROOT = Path(cfg["paths"]["data_root"])

OUTDIR_FIGURES = Path(cfg["paths"]["outdir_figures"])
OUTDIR_FIGURES.mkdir(parents=True, exist_ok=True)

OUTDIR_ANNDATA = Path(cfg["paths"]["outdir_anndata"])
OUTDIR_ANNDATA.mkdir(parents=True, exist_ok=True)

OUTDIR_UMAP = OUTDIR_FIGURES / "UMAP"
OUTDIR_UMAP.mkdir(parents=True, exist_ok=True)
OUTDIR_DEG = Path(cfg["paths"]["outdir_deg"])
OUTDIR_DEG.mkdir(parents=True, exist_ok=True)

sc.settings.figdir = str(OUTDIR_UMAP)

#  Load ordered aliases 
ordered_aliases = yaml.safe_load(open("configs/samples.template.yaml"))["ordered_aliases"]

#  Load alias -> real folder mapping (optional)
map_path = Path("configs/sample_map.yaml")
sample_map = {}
if map_path.exists():
    sample_map = yaml.safe_load(open(map_path))["sample_map"]

folder_names = [sample_map.get(a, a) for a in ordered_aliases]  # if no map, alias is folder name

#  Load metadata 
meta = pd.read_csv("configs/metadata.csv").set_index("alias")

#  Build file list 
sample_files = []
for alias, folder in zip(ordered_aliases, folder_names):
    f = DATA_ROOT / folder / "outs" / "filtered_feature_bc_matrix.h5"
    if f.exists():
        sample_files.append((alias, f))
    else:
        print(f"[WARN] Missing file for {alias}: {f}")

#  Read and annotate 
adatas = []
for alias, f in sample_files:
    print(f"Reading {alias}: {f}")

    adata = sc.read_10x_h5(str(f))
    adata.var_names_make_unique()

    # Attach metadata 
    adata.obs["Sample"] = alias

    # Attach metadata columns (from configs/metadata.csv)
    adata.obs["Barcode_ID"] = meta.loc[alias, "barcode_id"]
    adata.obs["SamplePoolName"] = meta.loc[alias, "sample_pool_name"]
    adata.obs["Species"] = meta.loc[alias, "species"]
    adata.obs["PrivateGeneticData"] = meta.loc[alias, "private_genetic_data"]
    adata.obs["Condition"] = meta.loc[alias, "condition"]

    # Optional: derive Genotype/Treatment/Full_Condition automatically
    cond = meta.loc[alias, "condition"]
    adata.obs["Genotype"] = "WT" if cond.startswith("CTRL") else "KO"
    adata.obs["Treatment"] = "IB" if cond in ["KOtreat", "CTRLTreat"] else "Non-treated"
    adata.obs["Full_Condition"] = adata.obs["Genotype"] + " " + adata.obs["Treatment"]

    adatas.append(adata)

In [ ]:
adata = sc.concat(adatas, axis=0)
adata.obs_names_make_unique()
adata

### AnnData object summary
```text
AnnData object with n_obs × n_vars = 22431 × 38606

In [ ]:
adata.write_h5ad(OUTDIR_ANNDATA / "adataConc.h5ad")

## Quality control - Gene Annotation

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))

## Computation of QC-metrics 

In [ ]:
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo"], inplace=True, log1p=True
)

In [ ]:
adata.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe()


In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"],
    jitter=0.3,
    multi_panel=True, 
    groupby="Sample",
    rotation = 90
)

In [ ]:
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_ribo")

## Cell and Gene filtering

In [ ]:
adata


## Quality control: cell and gene filtering

### Description
Cells were filtered based on standard quality control metrics to remove low-quality
cells and technical outliers prior to downstream analyses.

Filtering was applied sequentially at the cell level, followed by gene-level filtering.

### Cell filtering criteria
- Cells with **>20% mitochondrial counts** were removed.
- Cells expressing **fewer than 1,000 genes** were removed.
- Cells with low **total UMI counts** were filtered using a threshold on
  `log1p_total_counts`.




In [ ]:
# Remove cells with high mitochondrial content
adata1 = adata[adata.obs.pct_counts_mt < 20, :].copy()

# Remove cells with low gene complexity
adata2 = adata1[adata1.obs.n_genes_by_counts > 1000, :].copy()

# Filter cells based on total UMI counts
adata = adata2[adata2.obs.log1p_total_counts > 8, :].copy()

# Remove lowly expressed genes
sc.pp.filter_genes(adata, min_counts=1000)

### AnnData object summary-  Post filtering
```text
AnnData object with n_obs × n_vars = 18945 × 16259

In [ ]:
adata

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"],
    jitter=0.4,
    multi_panel=True, groupby="Sample"
    ,rotation = 90
)

In [ ]:
adata.write(OUTDIR_ANNDATA / "adataFiltered.h5ad")

In [ ]:
adata = sc.read_h5ad(OUTDIR_ANNDATA / "adataFiltered.h5ad")/

In [ ]:
adata.layers["counts"] = adata.X.copy()


In [ ]:
adata_raw = adata.copy()

In [ ]:
adata_raw

In [ ]:
adata_raw.write_h5ad(OUTDIR_ANNDATA / "adata_filtered_rawcounts.h5ad")

In [ ]:
adata_raw  = sc.read_h5ad(OUTDIR_ANNDATA / "adata_filtered_rawcounts.h5ad")

## Data normalization and log-transformation

Gene expression counts were normalized on a per-cell basis to account for differences
in sequencing depth across cells, followed by log-transformation.

This normalized representation is used for downstream analyses such as dimensionality
reduction and clustering.


In [ ]:

# Create a copy of the raw AnnData object
adata_all_log = adata_raw.copy()

# Library-size normalization
sc.pp.normalize_total(adata_all_log, target_sum=1e4)

# Log-transformation
sc.pp.log1p(adata_all_log)

In [ ]:
adata_all_log.write_h5ad(OUTDIR_ANNDATA / "adata_allGene_logNorm.h5ad")

In [ ]:
adata_all_log = sc.read_h5ad(OUTDIR_ANNDATA / "adata_allGene_logNorm.h5ad")

## Highly variable gene (HVG) selection
Highly variable genes were identified on the log-normalized dataset to retain genes
that capture the largest amount of biological variability across cells.
Only the selected highly variable genes were kept for clustering analysis

In [ ]:
##preparazione per calcolo di HVG su dataset log-normalizzato     
adata_hvg_log  = adata_all_log.copy()

In [ ]:
##Computing high variable genes on dataset log-normalizzato
sc.pp.highly_variable_genes(
    adata_hvg_log,
    n_top_genes =  3000,
    subset = True
)


## Dimensionality reduction and clustering strategy

### Description
After selecting highly variable genes, dimensionality reduction was performed using
Principal Component Analysis (PCA) to capture the main sources of variation in the data
while reducing noise and computational complexity.

The resulting low-dimensional representation was then used to build a cell–cell
neighborhood graph and perform graph-based clustering.

### Principal Component Analysis
PCA was computed using the highly variable, log-normalized gene expression matrix.
A total of **50 principal components (PCs)** were calculated to capture the majority
of variance in the dataset.

The cumulative explained variance was evaluated to assess how much biological signal
was retained by the selected number of PCs.

### Clustering and PC robustness assessment
Graph-based clustering was performed using the **Leiden algorithm** on neighborhood
graphs constructed with different numbers of PCs.

To assess the robustness of clustering with respect to the choice of dimensionality,
clustering solutions obtained using **20 to 24 PCs** were compared.

Cluster stability was quantified using the **Adjusted Rand Index (ARI)** between
clustering results obtained with adjacent numbers of PCs.

This procedure allows the identification of a range of PCs for which cluster
assignments are stable and not driven by arbitrary parameter choices.

### Reproducibility
All steps involving stochastic components were performed using fixed random seeds
to ensure reproducibility of the PCA and clustering results.


In [ ]:
from sklearn.decomposition import PCA
pca_log = PCA(n_components=50)
adata_log = adata_hvg_log.copy()

In [ ]:
# PCA
pca_log = PCA(n_components=50)
adata_log.obsm["X_pca"] = pca_log.fit_transform(adata_log.X)

print("LOG – varianza spiegata (ratio, primi 21 PC):")
print(pca_log.explained_variance_ratio_[:21].sum())
print("LOG – varianza totale spiegata dai 50 PC:", pca_log.explained_variance_ratio_.sum())


<p>
    Choosing to continue wiht 21pcs. The difference between 21 and 50 PCs is small
</p>

In [ ]:

import numpy as np
import random
from sklearn.metrics import adjusted_rand_score

np.random.seed(0)
random.seed(0)


max_pcs = 30
labels_by_n = {}

for n in range(20, 25):   # 20,21,22,23,24
    sc.pp.neighbors(adata_log, n_pcs=n, use_rep='X_pca')
    sc.tl.leiden(
        adata_log,
        key_added=f'leiden_{n}',
        resolution=0.5,
        random_state=0   #Riproducible Leiden
    )
    labels_by_n[n] = adata_log.obs[f'leiden_{n}'].astype(int).values

# compute ARI between adjacent n
aris = []
pcs = sorted(labels_by_n.keys())
for i in range(len(pcs) - 1):
    a = labels_by_n[pcs[i]]
    b = labels_by_n[pcs[i + 1]]
    aris.append(adjusted_rand_score(a, b))

for n, ari in zip(pcs[:-1], aris):
    print(f"ARI between {n} and {n+1} PCs: {ari:.3f}")


In [ ]:

import numpy as np
import random
from sklearn.metrics import adjusted_rand_score

np.random.seed(0)
random.seed(0)


max_pcs = 30
labels_by_n = {}

for n in range(20, 25):   # 20,21,22,23,24
    sc.pp.neighbors(adata_log, n_pcs=n, use_rep='X_pca')
    sc.tl.leiden(
        adata_log,
        key_added=f'leiden_{n}',
        resolution=0.2,
        random_state=0   #No stoc
    )
    labels_by_n[n] = adata_log.obs[f'leiden_{n}'].astype(int).values

# compute ARI 
aris = []
pcs = sorted(labels_by_n.keys())
for i in range(len(pcs) - 1):
    a = labels_by_n[pcs[i]]
    b = labels_by_n[pcs[i + 1]]
    aris.append(adjusted_rand_score(a, b))

for n, ari in zip(pcs[:-1], aris):
    print(f"ARI between {n} and {n+1} PCs: {ari:.3f}")


In [ ]:
#adata_log is -normalized - filtered for genes - 

adata_log.write_h5ad(OUTDIR_ANNDATA / "adataHvgNormalizedPostAri.h5ad")

In [ ]:
adata_log  = sc.read_h5ad(OUTDIR_ANNDATA "adataHvgNormalizedPostAri.h5ad")


## Clustering parameter optimization

### Description
To select robust and data-driven parameters for graph-based clustering, a systematic
exploration of the Leiden algorithm parameters was performed.

In particular, the effects of varying the number of nearest neighbors (**KNN**) and
the clustering **resolution** were evaluated while keeping the number of principal
components fixed.

### Parameter grid exploration
Clustering was performed across a grid of parameters defined by:
- multiple values of **KNN** (number of nearest neighbors used to build the graph)
- multiple values of **resolution** (controlling cluster granularity in Leiden)

For each parameter combination, a new neighborhood graph was constructed and Leiden
clustering was applied.

### Cluster quality assessment
To quantitatively assess clustering quality, two complementary internal validation
metrics were computed:
- **Silhouette score**, measuring cluster separation and cohesion
- **Davies–Bouldin index**, measuring cluster compactness and overlap

Both metrics were computed on the PCA representation used for clustering.

In addition, the **number of clusters** obtained for each parameter combination was
recorded to monitor changes in cluster granularity.


This parameter scan enables the identification of clustering configurations that:
- maximize cluster separation and compactness,
- avoid over-fragmentation or excessive merging of clusters,
- provide a stable and interpretable clustering solution.

The final clustering parameters were selected based on a balance between these
quantitative metrics and biological interpretability.


In [ ]:
import pandas as pd
from sklearn.metrics import silhouette_score, davies_bouldin_score

def scan_leiden_params(
    adata,
    knn_list = range(5, 45, 5),        # [5,10,15,...,40]
    res_list = [x/100 for x in range(50, 150, 10)],  # [0.5,0.6,...,1.4]
    n_pcs = 21,
    use_rep = "X_pca",
    seed = 0,
    key_prefix = "leiden_tmp"
):
   
    adata = adata.copy()   
    sc.settings.verbosity = 0
    sc.settings.seed = seed

    results = []

    for k in knn_list:
        sc.pp.neighbors(adata, n_neighbors=k, n_pcs=n_pcs, use_rep=use_rep)

        for res in res_list:
            key = f"{key_prefix}_k{k}_r{res}"

            sc.tl.leiden(adata, resolution=res, key_added=key, random_state=seed)

            labels = adata.obs[key].astype(str)

            sil = silhouette_score(adata.obsm[use_rep], labels)
            db  = davies_bouldin_score(adata.obsm[use_rep], labels)

            n_clust = labels.nunique()

            results.append({
                "KNN": k,
                "resolution": res,
                "number_of_clusters": n_clust,
                "sil": sil,
                "davie_bould": db,
            })

    results_df = pd.DataFrame(results)
    return results_df

In [ ]:
results_df = scan_leiden_params(
    adata_log,
    knn_list=range(5,45,5),
    res_list=[x/100 for x in range(50,150,10)],
    n_pcs= 21,
    use_rep="X_pca",
    seed = 0
)
    

In [ ]:
results_df_n_pcs = scan_leiden_params(
    adata_log,
    knn_list=range(5,45,5),
    res_list=[x/100 for x in range(50,150,10)],
    n_pcs= 50,
    use_rep="X_pca",
    seed = 0
)
    

In [ ]:
##Return sub_sorted Dataset

def orderScanLeidenParams(df):
    sub = df[
    (df["number_of_clusters"] >=8) & 
    (df["number_of_clusters"] <=30)]

    sub_sorted = sub.sort_values(
    by = ["sil", "davie_bould"],
    ascending = [False, True])

    return sub_sorted


In [ ]:
results_df_minus_res_ = scan_leiden_params(
    adata_log,
    knn_list=range(5,45,5),
    res_list=[x/100 for x in range(20,60,10)], #0.3, 0.4, 0.5
    n_pcs= 21,
    use_rep="X_pca",
    seed = 0
)

In [ ]:
#Choosing 0.2 resolution seeing the balance between sil and davie bould and a balance on KNN 

#Considering at least 9 cluster and a balance o

results_df_minus_res_.sort_values(by="sil", ascending =False).head(30)

##  Cluster analysis using Leiden

In [ ]:

###These are the parameters that are used to build clusters and beyond
sc.pp.neighbors(adata_log,n_neighbors=15,n_pcs=21,use_rep="X_pca")
sc.tl.umap(adata_log,random_state=0)
sc.tl.leiden(adata_log,resolution=0.2,   key_added="leiden_final",random_state=0)
sc.pl.umap(adata_log,color=["leiden_final","Full_Condition"],wspace = 0.4,show=False)



## Annotion using Cytetype

In [ ]:
from cytetype import CyteType

In [ ]:
#Going to do annotation 
annotator = CyteType(adata_log, group_key="leiden_final")

In [ ]:
adata_log1 = adata_log

In [ ]:
adata_log1 = annotator.run(
    study_context="Human Kidney",
)

In [ ]:
adata_log1.write_h5ad(OUTDIR_ANNDATA /"adata_Cytetype.h5ad")

In [ ]:
adata_log1 = sc.read_h5ad(OUTDIR_ANNDATA / "adata_Cytetype.h5ad")

In [ ]:
adata_log1.obs["cytetype_annotation_leiden_final"].cat.categories

In [ ]:
adata_log1.obs["cytetype_annotation_leiden_final"] = (
    adata_log1.obs["cytetype_annotation_leiden_final"].astype("category")
)



In [ ]:

## Changing colors of the UMAP clusters

import scanpy as sc


cell_types = adata_log1.obs["cytetype_annotation_leiden_final"].cat.categories

original_colors = [
    '#1f77b4',  # Distal Nephron Epithelial Cell
    '#ff7f0e',  # Immature Neuron (cerebellar granule)
    '#2ca02c',  # Immature Neuron (posterior glutamatergic)
    '#d62728',  # Kidney Interstitial Fibroblast
    '#9467bd',  # Myogenic Progenitor Cell
    '#8c564b',  # Neural Progenitor Cell
    '#e377c2',  # Podocyte
    '#7f7f7f',  # Proliferating Mesenchymal Cell
    '#bcbd22',  # Proximal Tubule Epithelial Cell
]   

grey_override = {
    'Immature Neuron (cerebellar granule)': '#d9d9d9',
    'Immature Neuron (posterior glutamatergic)': '#bdbdbd',
    'Neural Progenitor Cell': '#969696',
    'Proliferating Mesenchymal Cell': '#636363',
    'Myogenic Progenitor Cell': '#808080'
}

final_colors = []
for ct, col in zip(cell_types, original_colors):
    final_colors.append(grey_override.get(ct, col))

adata_log1.uns["cytetype_annotation_leiden_final_colors"] = final_colors


In [ ]:
sc.pl.embedding(adata_log1, basis='umap', color=f'cytetype_annotation_{"leiden_final"}',save = "UMAP_grey.png")

In [ ]:
sc.pl.embedding(adata_log1, basis='umap', color=f'cytetype_annotation_{"leiden_final"}',save="umap_leiden_res0_2_cytetype_1.png")


## Clusterization by Type

In [ ]:

##


import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

color_col      = "cytetype_annotation_leiden_final" 
condition_col  = "Full_Condition"                    

if f"{color_col}_colors" not in adata_log1.uns:
    sc.pl.embedding(
        adata_log1,
        basis="umap",
        color=color_col,
        show=False,
        legend_loc=None, 
    )

cell_types      = adata_log1.obs[color_col].cat.categories
celltype_colors = adata_log1.uns[f"{color_col}_colors"]

conditions = adata_log1.obs[condition_col].dropna().unique().tolist()
n_cond = len(conditions)
print("Condizioni trovate:", conditions)

n_cols = min(4, n_cond)
n_rows = int(np.ceil(n_cond / n_cols))

fig, axs = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4 * n_cols + 3, 4 * n_rows),  
)
axs = np.array(axs).reshape(-1)

for ax, cond in zip(axs, conditions):
    adata_sub = adata_log1[adata_log1.obs[condition_col] == cond].copy()
    
    sc.pl.embedding(
        adata_sub,
        basis="umap",
        color=color_col,
        ax=ax,
        show=False,
        legend_loc=None,      
    )
    
    ax.set_title(str(cond), fontsize=8)

for ax in axs[len(conditions):]:
    ax.axis("off")

handles = [
    Patch(color=col, label=ct)
    for ct, col in zip(cell_types, celltype_colors)
]

fig.legend(
    handles=handles,
    title="Cell types",
    loc="center right",
    bbox_to_anchor=(1.20, 0.5),
    fontsize=9,
    title_fontsize=7,
)

plt.subplots_adjust(right=0.82) 
plt.tight_layout()
plt.show()

# 5) Salva la figura
fig.savefig(
    OUTDIR_UMAP / "umap_cytetype_split_by_FullCondition_with_CellTypeLegend_Grey.png",
    dpi=300,
    bbox_inches="tight",
)


## Clusterization by Sample

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
import scanpy as sc

color_col     = "cytetype_annotation_leiden_final"
sample_col    = "Sample"
condition_col = "Full_Condition"

sample_to_cond = (
    adata_log1.obs[[sample_col, condition_col]]
    .drop_duplicates()
    .set_index(sample_col)[condition_col]
    .to_dict()
)

samples = list(sample_to_cond.keys())
print("Campioni:", samples)

n_samp = len(samples)
n_cols = min(4, n_samp)              
n_rows = int(np.ceil(n_samp / n_cols))

fig, axs = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4 * n_cols + 3, 4 * n_rows),   # +3 to give space to legend
)
axs = np.array(axs).reshape(-1)

for ax, samp in zip(axs, samples):
    cond = sample_to_cond[samp]
    
    adata_sub = adata_log1[adata_log1.obs[sample_col] == samp].copy()
    
    sc.pl.embedding(
        adata_sub,
        basis="umap",
        color=color_col,
        ax=ax,
        show=False,
        legend_loc=None,         
    )
    
    ax.set_title(f"{samp}\n{cond}", fontsize=8)

for ax in axs[len(samples):]:
    ax.axis("off")


cell_types      = adata_log1.obs[color_col].cat.categories
celltype_colors = adata_log1.uns[f"{color_col}_colors"]

handles = [
    Patch(color=col, label=ct)
    for ct, col in zip(cell_types, celltype_colors)
]

fig.legend(
    handles=handles,
    title="Cell types",
    loc="center right",
    bbox_to_anchor=(1.20, 0.47),
    fontsize=9,
    title_fontsize=8,
)

plt.subplots_adjust(right=10.99)  
plt.tight_layout()
plt.show()

fig.savefig(
    OUTDIR_UMAP / "umap_cytetype_split_by_SampleID_with_CellTypeLegendGrey.png",
    dpi=300,
    bbox_inches="tight",
)

## Checking markers 

In [ ]:
genes_focus = ["CUBN", "LRP2", "PDZK1", "SLC5A12"]  # Marker Proximal. See differences with cluster 6 

sc.pl.violin(
    adata_all_log,
    genes_focus,
    groupby="leiden_final",
    stripplot=False,
    jitter=False,
    multi_panel=True,
)

In [ ]:

genes_focus = ["MECOM", "KCNJ16", "CLDN4", "WFDC2"]  # Marker Distal. See differences with cluster 0 
#https://www.proteinatlas.org/ENSG00000085276-MECOM/single+cell/kidney
#https://www.proteinatlas.org/ENSG00000153822-KCNJ16/single+cell
#Cluster 6 like a mix of distal and collectig ductal cells 
#https://www.biorxiv.org/content/10.1101/2025.06.10.658766v1.full
sc.pl.violin(
    adata_all_log,
    genes_focus,
    groupby="leiden_final",
    stripplot=False,
    jitter=False,
    multi_panel=True,
)

In [ ]:
##About cluster 1 
#https://www.proteinatlas.org/ENSG00000134853-PDGFRA/single+cell
genes_focus = ["PDGFRA", "VIM", "DCN", "THY1","TCF21","ACTA2"]
sc.pl.violin(
    adata_all_log,
    genes_focus,
    groupby="leiden_final",
    stripplot=False,
    jitter=False,
    multi_panel=True,
)


In [ ]:
##About cluster 7
#see also sc.pl.rank_genes_groups(adata_log, n_genes=30,sharey=False)
#Vimentin 
genes_focus = ["VIM"]
sc.pl.violin(
    adata_all_log,
    genes_focus,
    groupby="leiden_final",
    stripplot=False,
    jitter=False,
    multi_panel=True,
)


In [ ]:
##About cluster 4
#SOX2 is the key marker for neural progenitor cell
genes_focus = ["SOX2","PTPRZ1","NES","HES1"]
sc.pl.violin( 
    adata_all_log,
    genes_focus,
    groupby="leiden_final",
    stripplot=False,
    jitter=False,
    multi_panel=True,
)


In [ ]:
mapping = {
"0": "Proximal Tubule Epithelial Cell",
"1": "Kidney Interstitial Fibroblast",
"2": "Immature Neuron (cerebellar granule)",
"3": "Podocyte",
"4": "Neural Progenitor Cell",
"5": "Myogenic Progenitor Cell",
"6": "Distal Nephron Epithelial Cell",
"7": "Proliferating Mesenchymal Cell",
"8": "Immature Neuron (posterior glutamatergic)",
}

# Se leiden_final è numerico, converti a stringa prima
leiden_str = adata_all_log.obs["leiden_final"].astype(str)

adata_all_log.obs["celltype"] = leiden_str.map(mapping)
adata_log1.obs["celltype"] = leiden_str.map(mapping)

In [ ]:
# Copy of metadata from HVG object to full object t
adata_all_log.obs['leiden_final'] = adata_log1.obs['leiden_final'].reindex(adata_all_log.obs_names)
adata_all_log.obs['Cell_Types']   = adata_log1.obs['celltype'].reindex(adata_all_log.obs_names)

In [ ]:

##Checking order of cells before to assign 
set(adata_log1.obs_names) == set(adata_all_log.obs_names)

In [ ]:
##Checking for NaN values 
adata_all_log.obs[["leiden_final","Cell_Types"]].isna().sum()

In [ ]:
adata_all_log.write_h5ad(OUTDIR_ANNDATA / "adata_all_log_Complete_Annotated.h5ad")

In [ ]:
#Computing markers on all matrix 
sc.tl.rank_genes_groups(
    adata_all_log,
    groupby="leiden_final",
    method="wilcoxon",
    use_raw=False
)

In [ ]:
adata_all_logMarkers = adata_all_log

In [ ]:
del adata_all_logMarkers.uns["rank_genes_groups"]
#adata.obs["leiden_final"].unique()

In [ ]:
adata_all_logMarkers.obs["leiden_final"] = adata_all_logMarkers.obs["leiden_final"].map(mapping)
adata_all_logMarkers.obs["leiden_final"] = adata_all_logMarkers.obs["leiden_final"].astype("category")

In [ ]:
#Computing markers on all matrix using rank genes groups
sc.tl.rank_genes_groups(
    adata_all_logMarkers,
    groupby="leiden_final",
    method="wilcoxon",
    use_raw=False
)

In [ ]:
proximalTuble = sc.get.rank_genes_groups_df(adata_all_logMarkers,group="Proximal Tubule Epithelial Cell",)

In [ ]:
sc.pl.rank_genes_groups(adata_all_logMarkers,n_genes=25,sharey=False, save="_leiden_rank_genes1.png")

## Differential gene expression analysis

### Overview
Differential gene expression (DGE) analysis was performed **separately for each annotated cell type**
to avoid confounding effects driven by differences in cell-type composition across conditions.

Within each cell type, gene expression profiles were compared across biologically relevant
experimental conditions.

### Comparisons
For each cell type, the following pairwise comparisons were performed:

- **KO IB vs WT Non-treated**
- **WT IB vs WT Non-treated**
- **KO IB vs KO Non-treated**
- **KO Non-treated vs WT Non-treated**

Only cell types containing a sufficient number of cells in both groups were included
in the analysis.

### Cell filtering for DGE
To ensure statistical robustness, comparisons were performed only when **both conditions**
contained at least **10 cells** within the given cell type.

Comparisons not meeting this criterion were excluded from the analysis.

### Statistical test
Differential expression was computed using the **Wilcoxon rank-sum test**, as implemented
in Scanpy.

The test was applied to log-normalized gene expression values and performed independently
for each cell type and comparison.

### Output
For each valid comparison, a table of differentially expressed genes was generated and
annotated with:
- cell type
- condition and reference group
- number of cells per condition

All results were combined into a single dataset for downstream interpretation.


In [ ]:
import scanpy as sc
import pandas as pd

adata_DGE = adata_all_log  # full, normalized/log, with Cell_Types and Full_Condition

all_results = []

cell_types = adata_DGE.obs['Cell_Types'].unique()

comparisons = [
    ("KO IB", "WT Non-treated"),
    ("WT IB", "WT Non-treated"),
    ("KO IB", "KO Non-treated"),
    ("KO Non-treated","WT Non-treated"),
    ("KO IB","WT IB")
]

min_cells_per_condition = 10  

for cell_type in cell_types:
    print(f"\n Analyzing cell type: {cell_type}")
    
    # For cell type
    adata_celltype = adata_DGE[adata_DGE.obs['Cell_Types'] == cell_type].copy()
    
    for cond, ctrl in comparisons:
        print(f"   {cond} vs {ctrl}")
        
        mask = adata_celltype.obs['Full_Condition'].isin([cond, ctrl])
        adata_subset = adata_celltype[mask].copy()
        
        present_conditions = adata_subset.obs['Full_Condition'].unique()
        if not all(c in present_conditions for c in [cond, ctrl]):
            print(f"   Skipping: {cond} or {ctrl} not found in this cell type.")
            continue
        
        # Filter on # of cell
        counts_per_cond = adata_subset.obs['Full_Condition'].value_counts()
        print("    Cells per condition:\n", counts_per_cond.to_dict())
        
        if (counts_per_cond[cond] < min_cells_per_condition) or (counts_per_cond[ctrl] < min_cells_per_condition):
            print("   Skipping due to too few cells in at least one condition.")
            continue
        
        sc.tl.rank_genes_groups(
            adata_subset,
            groupby="Full_Condition",
            groups=[cond],
            reference=ctrl,
            method="wilcoxon",
            use_raw=False,   
        )
        
        df = sc.get.rank_genes_groups_df(adata_subset, group=cond)
        df['cell_type']  = cell_type
        df['condition']  = cond
        df['control']    = ctrl
        df['comparison'] = f"{cond}_vs_{ctrl}"
        df['n_cells_cond']  = counts_per_cond[cond]
        df['n_cells_ctrl']  = counts_per_cond[ctrl]
        
        all_results.append(df)

deg_results = pd.concat(all_results, ignore_index=True)

## Cell distribution across conditions

### Description
The number of cells per cell type and experimental condition was summarized using
a contingency table.

This overview was used to assess dataset balance and to identify cell types with
insufficient representation in specific conditions.

In [ ]:
pd.crosstab(adata_DGE.obs["Cell_Types"], adata_DGE.obs["Full_Condition"])

In [ ]:
pd.crosstab(adata_DGE.obs["Cell_Types"], adata_DGE.obs["Full_Condition"]).to_csv(OUTDIR_ANNDATA / "numberOfCellForEachCellTypeAndCondition.txt",sep = "\t")

In [ ]:

#Sanitizing name of files 
import os
from pathlib import Path

output_dir = OUTDIR_DEG

def _sanitize(s: str) -> str:
    
    return (
        s.replace(" ", "_")
         .replace("/", "-")
         .replace("\\", "-")
         .replace("(", "")
         .replace(")", "")
         .replace(":", "-")
    )

for df in all_results:
    comparison = str(df["comparison"].iloc[0])
    cell_type  = str(df["cell_type"].iloc[0])

    comparison_safe = _sanitize(comparison)
    cell_type_safe  = _sanitize(cell_type)

    file_name = output_dir / f"{comparison_safe}__{cell_type_safe}.xlsx"
    print(file_name)

    df.to_excel(file_name, header=True, index=False)